In [0]:

# ============================================================
# CELDA 1. LIMPIEZA DE PRONÓSTICOS METEOROLÓGICOS
# ============================================================

# Objetivo:
# Preparar los pronósticos de Open-Meteo para unirlos
# posteriormente con los vuelos de BTS.
#
# Periodo: febrero de 2024 a diciembre de 2025.
# Aeropuertos: los 30 seleccionados.
# Pronósticos: 24 horas de anticipación.
#
# Bronze permanece intacto.

from pyspark.sql import functions as F

TABLA_BRONZE = "workspace.bronze_flights.weather_forecasts_raw"
TABLA_TOP30 = "workspace.bronze_flights.weather_airports_top30"
TABLA_SILVER = "workspace.silver_flights.weather_clean"

# ------------------------------------------------------------
# 1. LEER LOS DATOS DE BRONZE
# ------------------------------------------------------------

df_bronze = spark.table(TABLA_BRONZE)

# Seleccionamos únicamente los aeropuertos definidos
# para el modelo y el periodo meteorológico acordado.
top30 = (
    spark.table(TABLA_TOP30)
    .select("airport")
    .distinct()
)

df_clima = (
    df_bronze
    .join(F.broadcast(top30), on="airport", how="inner")
    .filter(F.col("source_model") == "gfs_global")
    .filter(F.col("forecast_lead_hours") == 24)
    .filter(
        (F.col("source_year") == 2025) |
        (
            (F.col("source_year") == 2024) &
            (F.col("source_month") >= 2)
        )
    )
)

# ------------------------------------------------------------
# 2. NORMALIZAR LOS TIPOS DE DATOS
# ------------------------------------------------------------

# Las horas de Open-Meteo se solicitaron en UTC.
# Convertimos su representación de texto a timestamp
# manteniendo UTC como zona de referencia.

spark.conf.set("spark.sql.session.timeZone", "UTC")

df_limpio = (
    df_clima
    .select(
        "airport",
        F.col("latitude").cast("double").alias("latitude"),
        F.col("longitude").cast("double").alias("longitude"),
        F.to_timestamp(
            "forecast_valid_time_utc",
            "yyyy-MM-dd'T'HH:mm"
        ).alias("weather_time_utc"),
        F.col("temperature_2m_previous_day1")
         .cast("double").alias("temperature_2m"),
        F.col("precipitation_previous_day1")
         .cast("double").alias("precipitation"),
        F.col("wind_speed_10m_previous_day1")
         .cast("double").alias("wind_speed_10m"),
        F.col("relative_humidity_2m_previous_day1")
         .cast("double").alias("relative_humidity_2m"),
        F.col("cloud_cover_previous_day1")
         .cast("double").alias("cloud_cover"),
        F.col("pressure_msl_previous_day1")
         .cast("double").alias("pressure_msl"),
        "source_model",
        "forecast_lead_hours",
        "source_year",
        "source_month"
    )
)

# ------------------------------------------------------------
# 3. VALIDAR CAMPOS ESENCIALES
# ------------------------------------------------------------

columnas_necesarias = [
    "airport",
    "weather_time_utc",
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

# Comprobamos que las columnas principales no
# tengan valores faltantes después de convertirlas.
nulos = df_limpio.agg(*[
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in columnas_necesarias
]).first().asDict()

print("NULOS EN VARIABLES METEOROLÓGICAS")

for columna, cantidad in nulos.items():
    print(f"{columna}: {cantidad}")

if any(cantidad > 0 for cantidad in nulos.values()):
    raise ValueError(
        "Se encontraron valores nulos en campos esenciales."
    )

# ------------------------------------------------------------
# 4. ELIMINAR DUPLICADOS
# ------------------------------------------------------------

# Cada aeropuerto debe tener un solo registro por hora,
# modelo meteorológico y horizonte del pronóstico.
df_limpio = df_limpio.dropDuplicates([
    "airport",
    "weather_time_utc",
    "source_model",
    "forecast_lead_hours"
])

# ------------------------------------------------------------
# 5. GUARDAR EN SILVER
# ------------------------------------------------------------

# Reemplazamos únicamente la tabla Silver derivada.
# No sobrescribimos la tabla original de Bronze.

(
    df_limpio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER)
)

# ------------------------------------------------------------
# 6. VALIDAR EL RESULTADO
# ------------------------------------------------------------

df_silver = spark.table(TABLA_SILVER)

total = df_silver.count()
aeropuertos = df_silver.select("airport").distinct().count()

periodos = (
    df_silver
    .select("airport", "source_year", "source_month")
    .distinct()
    .count()
)

print("\n" + "=" * 50)
print("RESULTADO SILVER METEOROLÓGICO")
print("=" * 50)

print("Registros limpios:", f"{total:,}")
print("Aeropuertos:", aeropuertos)
print("Periodos aeropuerto-mes:", periodos)
print("Tabla:", TABLA_SILVER)

if total == 504000 and aeropuertos == 30 and periodos == 690:
    print("VALIDACIÓN EXITOSA: Silver meteorológico listo.")
else:
    print("REVISAR: los registros no coinciden con lo esperado.")

display(df_silver.limit(10))
